# Phase 1 — Exploratory Data Analysis (EDA)

**CSC3109 Machine Learning | Group 30**

Before we train any model, we need to deeply understand our dataset. This notebook answers:

1. How many images do we have per category?
2. What do these aerial images actually look like?
3. What are the pixel statistics (brightness, colour distribution)?
4. Are the images consistent in size and format?
5. Why are these 4 categories visually confusable?
6. How do we create a clean train/validation split?

**Dataset:** `ml training data/set 30/`  
**Categories:** oil_well · solar_panel · transformer_station · wastewater_treatment_plant

---
## Step 0 — Imports

We import all the libraries we need up front.

- `os` / `pathlib` — navigate the file system to find our images
- `PIL` (Pillow) — open and read image files
- `numpy` — numerical computations (mean, std, histograms)
- `matplotlib` / `seaborn` — plotting and visualisation
- `pandas` — organise statistics in a table
- `random` — randomly select images and create splits
- `shutil` — copy files to new folders

In [ ]:
import os
import random
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image

# Makes plots look clean and professional
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

# Fix random seed so results are reproducible — anyone who re-runs this
# notebook gets the exact same train/val split
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print('All imports successful.')

---
## Step 1 — Define Paths

We set up the folder paths once here so we never have to hardcode them again.

- `DATA_ROOT` — where the raw dataset lives
- `TRAIN_DIR` / `VAL_DIR` — where we will put the organised train and validation splits

In [ ]:
# Root of the repository (one level up from this notebook)
REPO_ROOT = Path('..').resolve()

# Raw dataset — all 700 images per category live here
DATA_ROOT = REPO_ROOT / 'ml training data' / 'set 30'

# Destination for organised splits
TRAIN_DIR = REPO_ROOT / 'data' / 'train'
VAL_DIR   = REPO_ROOT / 'data' / 'val'

# The 4 categories in our dataset
CLASS_NAMES = sorted([d.name for d in DATA_ROOT.iterdir() if d.is_dir()])

print(f'Repository root : {REPO_ROOT}')
print(f'Dataset root    : {DATA_ROOT}')
print(f'Categories found: {CLASS_NAMES}')

---
## Step 2 — Count Images Per Category

Before anything else, we verify that each category has exactly 700 images and that there are no corrupted files.

In [ ]:
SUPPORTED_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}

records = []
for cls in CLASS_NAMES:
    cls_dir = DATA_ROOT / cls
    image_paths = [p for p in cls_dir.iterdir() if p.suffix.lower() in SUPPORTED_EXTS]
    
    # Attempt to open each image to catch corrupted files
    corrupted = []
    valid_paths = []
    for p in image_paths:
        try:
            with Image.open(p) as img:
                img.verify()  # checks file integrity without loading pixels
            valid_paths.append(p)
        except Exception:
            corrupted.append(p.name)
    
    records.append({
        'Category': cls,
        'Total Files': len(image_paths),
        'Valid Images': len(valid_paths),
        'Corrupted': len(corrupted),
    })
    
    if corrupted:
        print(f'  WARNING — {cls} has {len(corrupted)} corrupted file(s): {corrupted}')

count_df = pd.DataFrame(records)
print(count_df.to_string(index=False))
print(f'\nTotal images: {count_df["Valid Images"].sum()}')

In [ ]:
# Visualise class distribution as a bar chart
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(count_df['Category'], count_df['Valid Images'],
              color=['#4C72B0', '#DD8452', '#55A868', '#C44E52'], width=0.5)

ax.set_title('Image Count per Category', fontsize=14, fontweight='bold')
ax.set_ylabel('Number of Images')
ax.set_xlabel('Category')
ax.set_ylim(0, 800)
ax.axhline(700, linestyle='--', color='grey', alpha=0.6, label='Expected (700)')

for bar in bars:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
            str(int(bar.get_height())), ha='center', va='bottom', fontweight='bold')

# Wrap long category names for readability
ax.set_xticklabels([c.replace('_', '\n') for c in count_df['Category']], fontsize=9)
ax.legend()
plt.tight_layout()
plt.savefig('../report/eda_class_distribution.png', bbox_inches='tight')
plt.show()
print('Saved: report/eda_class_distribution.png')

---
## Step 3 — Visual Inspection: Sample Images

This is the most important step. We look at actual images from each category with our own eyes.

**Key question to think about while looking:**  
Why would a model confuse these categories? What features do they share?

In [ ]:
SAMPLES_PER_CLASS = 5  # number of images to display per category

fig, axes = plt.subplots(len(CLASS_NAMES), SAMPLES_PER_CLASS,
                          figsize=(SAMPLES_PER_CLASS * 2.5, len(CLASS_NAMES) * 2.5))

fig.suptitle('Sample Aerial Images by Category', fontsize=15, fontweight='bold', y=1.01)

for row_idx, cls in enumerate(CLASS_NAMES):
    cls_dir = DATA_ROOT / cls
    all_images = sorted([p for p in cls_dir.iterdir() if p.suffix.lower() in SUPPORTED_EXTS])
    samples = random.sample(all_images, SAMPLES_PER_CLASS)

    for col_idx, img_path in enumerate(samples):
        ax = axes[row_idx][col_idx]
        img = Image.open(img_path).convert('RGB')
        ax.imshow(img)
        ax.axis('off')
        if col_idx == 0:
            ax.set_ylabel(cls.replace('_', '\n'), fontsize=9, fontweight='bold',
                          rotation=0, labelpad=80, va='center')

plt.tight_layout()
plt.savefig('../report/eda_sample_images.png', bbox_inches='tight')
plt.show()
print('Saved: report/eda_sample_images.png')

### Observation: Why Are These Categories Confusable?

*Fill this in after viewing the images above. Consider:*

- **oil_well vs transformer_station:** Both may appear as clusters of grey/metallic structures on bare land. Oil wells have a circular pad; transformer stations have a grid-like pattern of insulators and wiring.
- **solar_panel vs wastewater_treatment_plant:** Both feature large flat rectangular or circular structures. Solar arrays are dark blue/black in colour; wastewater tanks are circular with liquid inside.
- **Common confusors:** Similar overhead viewing angle, similar muted colour palettes (grey, brown), similar industrial scale.

*Update this cell with your own observations after running the cell above.*

---
## Step 4 — Image Metadata: Dimensions and File Size

Neural networks require all input images to be the same size. Here we check whether our images are consistent, and what size they are. We will also flag any unusual images.

In [ ]:
metadata_records = []

for cls in CLASS_NAMES:
    cls_dir = DATA_ROOT / cls
    image_paths = sorted([p for p in cls_dir.iterdir() if p.suffix.lower() in SUPPORTED_EXTS])
    
    for p in image_paths:
        with Image.open(p) as img:
            w, h = img.size
            mode = img.mode  # 'RGB', 'L' (grayscale), 'RGBA', etc.
        file_size_kb = p.stat().st_size / 1024
        metadata_records.append({
            'category': cls,
            'filename': p.name,
            'width': w,
            'height': h,
            'mode': mode,
            'file_size_kb': round(file_size_kb, 1),
        })

meta_df = pd.DataFrame(metadata_records)

# Summary statistics
print('=== Image Dimensions ===')
print(meta_df.groupby('category')[['width', 'height']].describe().round(1))

print('\n=== Unique Sizes (W×H) ===')
size_counts = meta_df.groupby(['width', 'height']).size().reset_index(name='count')
print(size_counts.sort_values('count', ascending=False).to_string(index=False))

print('\n=== Colour Modes ===')
print(meta_df['mode'].value_counts().to_string())

print('\n=== File Size (KB) ===')
print(meta_df.groupby('category')['file_size_kb'].describe().round(1))

In [ ]:
# Flag non-RGB images — these need special handling during preprocessing
non_rgb = meta_df[meta_df['mode'] != 'RGB']
if len(non_rgb) > 0:
    print(f'WARNING: {len(non_rgb)} non-RGB images found:')
    print(non_rgb[['category', 'filename', 'mode']].to_string(index=False))
else:
    print('All images are RGB — no special handling needed.')

# Check for non-square images
non_square = meta_df[meta_df['width'] != meta_df['height']]
if len(non_square) > 0:
    print(f'\nWARNING: {len(non_square)} non-square images found.')
    print(non_square[['category', 'filename', 'width', 'height']].head(10).to_string(index=False))
else:
    print('All images are square — good.')

---
## Step 5 — Pixel Value Analysis

We look at the distribution of pixel brightness across each category.

**Why this matters:**  
- If categories have very different colour profiles, a model can learn to distinguish them by colour alone (easy).
- If colour profiles are similar, the model must rely on shape and texture (harder — but more meaningful learning).
- The mean and std per channel tells us what normalisation values to use in training.

**Note:** Loading all 2800 images to compute statistics takes ~1–2 minutes. We sample 100 images per class to speed this up.

In [ ]:
SAMPLE_SIZE = 100  # images per class to sample for pixel analysis

channel_stats = {}  # category → {'R': [...], 'G': [...], 'B': [...]}

for cls in CLASS_NAMES:
    cls_dir = DATA_ROOT / cls
    all_paths = sorted([p for p in cls_dir.iterdir() if p.suffix.lower() in SUPPORTED_EXTS])
    sampled = random.sample(all_paths, min(SAMPLE_SIZE, len(all_paths)))
    
    r_vals, g_vals, b_vals = [], [], []
    for p in sampled:
        img = np.array(Image.open(p).convert('RGB'), dtype=np.float32)
        r_vals.append(img[:, :, 0].mean())
        g_vals.append(img[:, :, 1].mean())
        b_vals.append(img[:, :, 2].mean())
    
    channel_stats[cls] = {'R': r_vals, 'G': g_vals, 'B': b_vals}
    print(f'{cls}: R={np.mean(r_vals):.1f}±{np.std(r_vals):.1f}  '
          f'G={np.mean(g_vals):.1f}±{np.std(g_vals):.1f}  '
          f'B={np.mean(b_vals):.1f}±{np.std(b_vals):.1f}')

print('\nPixel values are in [0, 255]. Higher = brighter.')

In [ ]:
# Plot per-channel mean brightness distributions as violin plots
colors = {'R': '#e74c3c', 'G': '#2ecc71', 'B': '#3498db'}

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
fig.suptitle('Per-Channel Mean Pixel Brightness by Category', fontsize=13, fontweight='bold')

for ax, channel in zip(axes, ['R', 'G', 'B']):
    data = [channel_stats[cls][channel] for cls in CLASS_NAMES]
    parts = ax.violinplot(data, positions=range(len(CLASS_NAMES)), showmedians=True)
    
    for pc in parts['bodies']:
        pc.set_facecolor(colors[channel])
        pc.set_alpha(0.7)
    parts['cmedians'].set_color('black')
    
    ax.set_title(f'{channel} Channel', color=colors[channel], fontweight='bold')
    ax.set_xticks(range(len(CLASS_NAMES)))
    ax.set_xticklabels([c.replace('_', '\n') for c in CLASS_NAMES], fontsize=8)
    ax.set_ylabel('Mean Pixel Value (0–255)' if channel == 'R' else '')

plt.tight_layout()
plt.savefig('../report/eda_channel_brightness.png', bbox_inches='tight')
plt.show()
print('Saved: report/eda_channel_brightness.png')

In [ ]:
# Plot full pixel histogram for one sample image per category
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()
fig.suptitle('Pixel Value Histogram (one sample image per category)', fontsize=13, fontweight='bold')

channel_colors = {'R': '#e74c3c', 'G': '#2ecc71', 'B': '#3498db'}

for idx, cls in enumerate(CLASS_NAMES):
    cls_dir = DATA_ROOT / cls
    sample_path = sorted([p for p in cls_dir.iterdir() if p.suffix.lower() in SUPPORTED_EXTS])[0]
    img = np.array(Image.open(sample_path).convert('RGB'))
    
    ax = axes[idx]
    for ch_idx, (ch_name, ch_color) in enumerate(channel_colors.items()):
        ax.hist(img[:, :, ch_idx].flatten(), bins=64, range=(0, 256),
                color=ch_color, alpha=0.5, label=ch_name, density=True)
    
    ax.set_title(cls.replace('_', ' ').title(), fontweight='bold')
    ax.set_xlabel('Pixel Value')
    ax.set_ylabel('Density')
    ax.legend()

plt.tight_layout()
plt.savefig('../report/eda_pixel_histograms.png', bbox_inches='tight')
plt.show()
print('Saved: report/eda_pixel_histograms.png')

---
## Step 6 — Compute Dataset-Wide Mean and Std (for Normalisation)

During training we will normalise pixel values using the dataset's own mean and standard deviation per channel. These values are computed here once and then hardcoded into `src/dataset.py`.

**Why normalise?**  
Neural networks train faster and more stably when inputs have zero mean and unit variance. Without normalisation, pixels in the range [0, 255] cause very large initial activations, making gradients unstable.

**Note:** We sample 200 images per class to estimate these values (full computation would take ~5 minutes on CPU).

In [ ]:
NORM_SAMPLE = 200  # images per class to use for mean/std estimation

all_means = []  # will be a list of (R_mean, G_mean, B_mean) per image
all_stds  = []

for cls in CLASS_NAMES:
    cls_dir = DATA_ROOT / cls
    all_paths = sorted([p for p in cls_dir.iterdir() if p.suffix.lower() in SUPPORTED_EXTS])
    sampled = random.sample(all_paths, min(NORM_SAMPLE, len(all_paths)))
    
    for p in sampled:
        # Load image, convert to [0, 1] float
        img = np.array(Image.open(p).convert('RGB'), dtype=np.float32) / 255.0
        all_means.append(img.mean(axis=(0, 1)))  # shape: (3,)
        all_stds.append(img.std(axis=(0, 1)))    # shape: (3,)

mean_per_channel = np.array(all_means).mean(axis=0)
std_per_channel  = np.array(all_stds).mean(axis=0)

print('Dataset normalisation values (pixel range [0, 1]):')
print(f'  Mean: R={mean_per_channel[0]:.4f}  G={mean_per_channel[1]:.4f}  B={mean_per_channel[2]:.4f}')
print(f'  Std:  R={std_per_channel[0]:.4f}  G={std_per_channel[1]:.4f}  B={std_per_channel[2]:.4f}')
print()
print('ImageNet values (used as alternative):')
print('  Mean: R=0.485  G=0.456  B=0.406')
print('  Std:  R=0.229  G=0.224  B=0.225')
print()
print('We will use ImageNet values for transfer learning models (Approaches 2–5)')
print('and dataset-specific values for the custom CNN (Approach 1).')

---
## Step 7 — Create Train / Validation Split

The spec says we need **100 held-out validation images per category**. We select them randomly here and copy them into `data/val/`. The remaining 600 go into `data/train/`.

**Important rules:**
- The 100 validation images must NEVER be used during training
- The same random seed (42) must be used every time so the split is reproducible
- We copy the files (not move) to keep the original dataset intact

**Note:** If `data/train/` and `data/val/` already exist and are populated, this step is skipped to avoid overwriting.

In [ ]:
VAL_PER_CLASS = 100

# Check if split already exists
split_done = all(
    (VAL_DIR / cls).exists() and len(list((VAL_DIR / cls).iterdir())) > 0
    for cls in CLASS_NAMES
)

if split_done:
    print('Train/val split already exists — skipping.')
    print('Delete data/train/ and data/val/ to redo the split.')
else:
    print(f'Creating train/val split (val={VAL_PER_CLASS} per class, seed={RANDOM_SEED})...')
    
    split_summary = []
    for cls in CLASS_NAMES:
        cls_dir = DATA_ROOT / cls
        all_paths = sorted([p for p in cls_dir.iterdir() if p.suffix.lower() in SUPPORTED_EXTS])
        
        # Shuffle with fixed seed and take first VAL_PER_CLASS as validation
        random.shuffle(all_paths)
        val_paths   = all_paths[:VAL_PER_CLASS]
        train_paths = all_paths[VAL_PER_CLASS:]
        
        # Create destination directories
        (TRAIN_DIR / cls).mkdir(parents=True, exist_ok=True)
        (VAL_DIR / cls).mkdir(parents=True, exist_ok=True)
        
        # Copy files
        for p in train_paths:
            shutil.copy2(p, TRAIN_DIR / cls / p.name)
        for p in val_paths:
            shutil.copy2(p, VAL_DIR / cls / p.name)
        
        split_summary.append({
            'Category': cls,
            'Train': len(train_paths),
            'Val': len(val_paths),
            'Total': len(all_paths),
        })
    
    split_df = pd.DataFrame(split_summary)
    print('\nSplit complete:')
    print(split_df.to_string(index=False))
    print(f"\nTrain total: {split_df['Train'].sum()} images")
    print(f"Val total:   {split_df['Val'].sum()} images")

In [ ]:
# Verify the split counts
print('Verifying split...')
for cls in CLASS_NAMES:
    train_count = len(list((TRAIN_DIR / cls).iterdir()))
    val_count   = len(list((VAL_DIR / cls).iterdir()))
    status = '✓' if val_count == VAL_PER_CLASS else '✗ MISMATCH'
    print(f'  {cls}: train={train_count}  val={val_count}  {status}')

---
## Step 8 — Visualise Split as Stacked Bar Chart

In [ ]:
train_counts = [len(list((TRAIN_DIR / cls).iterdir())) for cls in CLASS_NAMES]
val_counts   = [len(list((VAL_DIR / cls).iterdir())) for cls in CLASS_NAMES]

x = np.arange(len(CLASS_NAMES))
fig, ax = plt.subplots(figsize=(9, 4))

bars_train = ax.bar(x, train_counts, label='Train', color='#4C72B0', width=0.5)
bars_val   = ax.bar(x, val_counts, bottom=train_counts, label='Validation', color='#DD8452', width=0.5)

ax.set_title('Train / Validation Split per Category', fontsize=13, fontweight='bold')
ax.set_ylabel('Number of Images')
ax.set_xticks(x)
ax.set_xticklabels([c.replace('_', '\n') for c in CLASS_NAMES], fontsize=9)
ax.legend()

# Annotate bars
for i in range(len(CLASS_NAMES)):
    ax.text(i, train_counts[i] / 2, str(train_counts[i]), ha='center', va='center',
            color='white', fontweight='bold', fontsize=10)
    ax.text(i, train_counts[i] + val_counts[i] / 2, str(val_counts[i]), ha='center',
            va='center', color='white', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.savefig('../report/eda_train_val_split.png', bbox_inches='tight')
plt.show()
print('Saved: report/eda_train_val_split.png')

---
## Step 9 — EDA Summary for Report

Run this cell to print a clean summary you can copy-paste into your report.

In [ ]:
dominant_dim = meta_df.groupby(['width', 'height']).size().idxmax()

print('=' * 55)
print('EDA SUMMARY — CSC3109 Group 30')
print('=' * 55)
print(f'Dataset:    Set 30 (4-class aerial imagery subset)')
print(f'Categories: {len(CLASS_NAMES)}')
for cls in CLASS_NAMES:
    print(f'  - {cls}')
print(f'\nImage count per category: 700')
print(f'Total raw images:         2,800')
print(f'Dominant image size:      {dominant_dim[0]}×{dominant_dim[1]} px')
print(f'Colour mode:              RGB')
print(f'\nTrain/val split (seed={RANDOM_SEED}):')
print(f'  Train: 600 per class = 2,400 total')
print(f'  Val:   100 per class =   400 total')
print(f'\nDataset-specific normalisation values:')
print(f'  Mean: [{mean_per_channel[0]:.4f}, {mean_per_channel[1]:.4f}, {mean_per_channel[2]:.4f}]')
print(f'  Std:  [{std_per_channel[0]:.4f}, {std_per_channel[1]:.4f}, {std_per_channel[2]:.4f}]')
print(f'\nImageNet normalisation (used for transfer learning):')
print(f'  Mean: [0.485, 0.456, 0.406]')
print(f'  Std:  [0.229, 0.224, 0.225]')
print('=' * 55)

---
## Phase 1 Complete

The following outputs are saved to `report/` and ready to include in the final report:

| File | Description |
|------|-------------|
| `eda_class_distribution.png` | Bar chart of image count per category |
| `eda_sample_images.png` | Grid of 5 sample images per category |
| `eda_channel_brightness.png` | Violin plots of R/G/B brightness per category |
| `eda_pixel_histograms.png` | Pixel value histograms per category |
| `eda_train_val_split.png` | Stacked bar of train/val counts |

**Next:** Phase 2 — Data Preparation and Augmentation (`notebooks/02_data_prep.ipynb`)